# Collect data

Downloads DNA's monthly DUA files into `data/raw/<year>/`, from 2016-01 up to the last month that has ended.

## Where the files come from

DNA publishes them on its public FTP server, as allowed by Ley 19.438, art. 43 (see `docs/configuracion-cliente-ftp.pdf` for DNA's access guide):

- Server `ftp.aduanas.gub.uy`, port 21, anonymous access.
- Folder `/DUA Diarios XML/<year>/`, with daily (`ddyyyymmdd.zip`) and monthly (`dmyyyymm.zip`) files. Only the monthly ones are used (see `0.1`).

## How the download works

- **Only months that have ended.** The current month's file is rewritten every day until the month ends, so it's left out.
- **Only what changed.** For each month, the server's file size and modification date are compared with the local copy; if they match, nothing is downloaded. The server's modification date is copied to the downloaded file, so this check also works on later runs.
- **`data/raw/` is never overwritten.** If DNA republishes a month already downloaded, the new version is saved next to the old one as `dmyyyymm_<server date>.zip`, and `0.1` uses the newest one.
- **Files are asked for by name.** The server blocks folder listings from many networks, so each month is queried directly (`SIZE` and `MDTM`, which don't need a listing). There's a short pause between requests to avoid overloading the server.

The code lives in `uruguay_dua_comex/pipeline.py`, shared with the automatic update (`python -m uruguay_dua_comex.update`).

In [ ]:
import logging

from tqdm.auto import tqdm

from uruguay_dua_comex.pipeline import connect, download_month, list_remote_months

logging.basicConfig(level=logging.INFO, format="%(message)s")

ftp = connect()
remote_months = list_remote_months(ftp)
for remote in tqdm(remote_months, desc="Checking months"):
    download_month(ftp, remote)
ftp.quit()